In [1]:
# importa as bilbiotecas

import requests as rq
import urllib3
import pandas as pd

In [2]:
#faz o request
# Evita warning de certificado SSL (para testes locais)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = 'https://api-comexstat.mdic.gov.br/cities'

headers = {
    'Accept': 'application/json',
    'Content-Type': 'application/json'
}

body = {
   "flow": "import",
   "monthDetail": True,
   "period":{
       "from": "2025-01",
       "to": "2025-12"
   },
   "filters":[
       {
           "filter":"heading",
           "values":[6401,6402,6403,6404,6405]
       }
   ],
   "details":[
       "country",
       "state",
       "heading",
       "city",
   ],
   "metrics":[
       "metricFOB",
       "metricKG"
   ]

}

response = rq.post(url, headers=headers, json=body, verify=False)

print("Status Code:", response.status_code)
print(response.text)


Status Code: 200
{"data":{"list":[{"noMunMinsgUf":"Extrema - MG","year":"2025","monthNumber":"02","country":"Vietn\u00e3","state":"Minas Gerais","headingCode":"6404","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de mat\u00e9rias t\u00eaxteis","metricFOB":"7748377","metricKG":"223993"},{"noMunMinsgUf":"Extrema - MG","year":"2025","monthNumber":"01","country":"Vietn\u00e3","state":"Minas Gerais","headingCode":"6404","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de mat\u00e9rias t\u00eaxteis","metricFOB":"7429002","metricKG":"209767"},{"noMunMinsgUf":"Extrema - MG","year":"2025","monthNumber":"03","country":"Indon\u00e9sia","state":"Minas Gerais","headingCode":"6404","heading":"Cal\u00e7ado com sola exterior de borracha, pl\u00e1stico, couro natural ou reconstitu\u00eddo e parte superior de mat\u00e9rias t\u00eaxteis","metricFOB":"724673

In [3]:
#transforma o retorno em json
json_dados = response.json()
print(json_dados)

{'data': {'list': [{'noMunMinsgUf': 'Extrema - MG', 'year': '2025', 'monthNumber': '02', 'country': 'Vietnã', 'state': 'Minas Gerais', 'headingCode': '6404', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de matérias têxteis', 'metricFOB': '7748377', 'metricKG': '223993'}, {'noMunMinsgUf': 'Extrema - MG', 'year': '2025', 'monthNumber': '01', 'country': 'Vietnã', 'state': 'Minas Gerais', 'headingCode': '6404', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de matérias têxteis', 'metricFOB': '7429002', 'metricKG': '209767'}, {'noMunMinsgUf': 'Extrema - MG', 'year': '2025', 'monthNumber': '03', 'country': 'Indonésia', 'state': 'Minas Gerais', 'headingCode': '6404', 'heading': 'Calçado com sola exterior de borracha, plástico, couro natural ou reconstituído e parte superior de matérias têxteis', 'metricFOB': '7246738', 'metricKG': '274676'}, {'noMunMinsgUf': 'Extrema -

In [4]:
#cria dataframe
normaliza_dados = pd.json_normalize(json_dados['data']['list'])
data_frame = pd.DataFrame(normaliza_dados)

In [5]:
#ajusta o tipo dos dados
data_frame = data_frame.astype({
    'year': int,
    'monthNumber': int,
    'metricFOB': float,
    'metricKG': float,
    'headingCode': int
})

In [6]:
#ordena os dados
data_frame = data_frame.sort_values(by='monthNumber', ascending=True)

In [7]:
#cria coluna dia
data_frame['dia'] = 1

In [8]:
#cria coluna data
data_frame['data'] = (data_frame['dia'].astype(str).str.zfill(2)+'/'+data_frame['monthNumber'].astype(str).str.zfill(2)+'/'+data_frame['year'].astype(str))

In [9]:
#ordena colunas
data_frame = data_frame[['year','monthNumber','dia','data','headingCode','heading','country','state','noMunMinsgUf','metricFOB','metricKG']]

In [10]:
#salva dados em excel
data_frame.to_excel('./dados_importacao.xlsx', header=True, index=False,sheet_name='importacao')